### Import libaries

In [71]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics.pairwise import sigmoid_kernel,linear_kernel,rbf_kernel,polynomial_kernel


#### Load Dataset

In [101]:
# Load the books data set
df1=pd.read_csv("Books.csv",encoding='latin-1')  
# Load the Users data set
df2=pd.read_csv("Users.csv",encoding='latin-1')
# Load the Ratings data set
df3=pd.read_csv("Ratings.csv",encoding='latin-1')

#### Preprocessing for Books Dataset df1

In [73]:
df1.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [74]:
df1.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [75]:
df1 = df1.dropna(subset=['Book-Author'])

In [76]:
df1.duplicated().sum()

np.int64(0)

In [98]:
df1.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            0
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [77]:
books=df1[['ISBN', 'Book-Title', 'Book-Author']]

In [99]:
books.head()

,ISBN,Book-Title,Book-Author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


#### Preprocessing for Users Dataset df2

In [102]:
df2.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [103]:
df2.isnull().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [104]:
# Convert 'NaN' string to actual NaN
df2['Age'] = df2['Age'].replace('NaN', np.nan)
# Fill missing age
df2['Age'] = df2['Age'].fillna(df2['Age'].median())
# Remove unrealistic ages
df2 = df2[(df2['Age'] >= 5) & (df2['Age'] <= 90)]

In [105]:
df2.isnull().sum()

User-ID     0
Location    0
Age         0
dtype: int64

In [106]:
df2[['city', 'state', 'country']] = df2['Location'].str.split(',', n=2, expand=True)

In [107]:
df2['city'] = df2['city'].str.strip()
df2['state'] = df2['state'].str.strip()
df2['country'] = df2['country'].str.strip()

In [108]:
df2 = df2.drop('Location', axis=1)

In [109]:
df2.head()

,User-ID,Age,city,state,country
0,1,32.0,nyc,new york,usa
1,2,18.0,stockton,california,usa
2,3,32.0,moscow,yukon territory,russia
3,4,17.0,porto,v.n.gaia,portugal
4,5,32.0,farnborough,hants,united kingdom


#### Preprocessing for Ratings Dataset df3

In [85]:
df3.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [86]:
df3.isnull().sum()

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

In [87]:
df3.duplicated().sum()

np.int64(0)

#### Merge Datasets

In [88]:
merge1=df3.merge(books,on='ISBN',how='inner')

In [89]:
df=merge1.merge(df2,on='User-ID',how='inner')

In [90]:
df.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Age,city,state,country
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,32.0,tyler,texas,usa
1,276726,0155061224,5,Rites of Passage,Judith Rae,32.0,seattle,washington,usa
2,276727,0446520802,0,The Notebook,Nicholas Sparks,16.0,h,new south wales,australia
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,16.0,rijeka,n/a,croatia
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,16.0,rijeka,n/a,croatia


In [110]:
df.isnull().sum()

User-ID        0
ISBN           0
Book-Rating    0
Book-Title     0
Book-Author    0
Age            0
city           0
state          0
country        0
dtype: int64

#### Filter

 We filter active users and popular items the reduce sparsity and improve recommendation quality.

In [111]:
# Keep active users & popular books
min_user_ID = 50
min_book_ratings = 50

active_users = df['User-ID'].value_counts()
active_users = active_users[active_users > min_user_ID].index

popular_books = df['ISBN'].value_counts()
popular_books = popular_books[popular_books > min_book_ratings].index

df_filtered = df[
    (df['User-ID'].isin(active_users)) &
    (df['ISBN'].isin(popular_books))
]

In [112]:
user_book_matrix = df_filtered.pivot_table(
    index='User-ID',
    columns='ISBN',
    values='Book-Rating'
).fillna(0)

In [113]:
user_book_matrix.head()

ISBN,000649840X,002026478X,0020442203,002542730X,0028604199,006000438X,0060008032,0060008776,006001203X,0060085444,...,1860492592,1878424319,1885171080,1931561648,3257228007,3257229534,3404148665,3423202327,3442541751,3492045170
User-ID,,,,,,,,,,,,,,,,,,,,,
243,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
254,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
507,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
